Pemisahan label netral dan positif/negatif


In [ ]:
"""
FASE : Pemisahan Berkas Review Manual (Polar vs Netral)
========================================================
MODE = "SPLIT"   -> dataset_review_polar.csv + dataset_review_netral.csv
MODE = "MERGE"   -> dataset_label_optimal.csv (setelah label_manual diisi)
MODE = "LAPORAN" -> laporan_split_review.md + 2 berkas PNG
"""

import logging
from datetime import datetime
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# ==================================================================
# KONFIGURASI
# ==================================================================
MODE = "LAPORAN"

FILE_INPUT = Path(
    r"G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_final.csv"
)
FOLDER_OUTPUT = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")

FILE_POLAR = FOLDER_OUTPUT / "dataset_review_polar.csv"
FILE_NETRAL = FOLDER_OUTPUT / "dataset_review_netral.csv"
FILE_OPTIMAL = FOLDER_OUTPUT / "dataset_label_optimal.csv"
FILE_LAPORAN = FOLDER_OUTPUT / "laporan_split_review.md"
FOLDER_VISUAL = FOLDER_OUTPUT / "hasil_visualisasi"

TEXT_COL = "text_clean_bert"
DATE_COL = "tanggal_wib"
KEY_COL = "row_key"
ALIAS_MODEL = ["mdhugol", "w11wo_roberta", "ayame_bertcahya"]

BOBOT_SEPAKAT = 0.5
BOBOT_CONF = 0.5
AMBANG_PERSENTIL = 25.0
SERTAKAN_SEMUA_KOLOM = False

DESIMAL_KOMA = True
BIN_HISTOGRAM = 30
DPI = 300

VALID_CLS = ("pos", "neu", "neg")
ENC = "utf-8-sig"
WARNA = {"pos": "#2ecc71", "neu": "#95a5a6", "neg": "#e74c3c", "ambigu": "#f39c12"}
NAMA_KELAS = {"pos": "Positif", "neu": "Netral", "neg": "Negatif", "ambigu": "Ambigu"}


# ==================================================================
# INTI
# ==================================================================
def muat() -> pd.DataFrame:
    if not FILE_INPUT.exists():
        raise FileNotFoundError(f"Tidak ditemukan: {FILE_INPUT}")
    df = pd.read_csv(FILE_INPUT, encoding_errors="replace")
    wajib = [TEXT_COL] + [f"label_{a}" for a in ALIAS_MODEL]
    hilang = [c for c in wajib if c not in df.columns]
    if hilang:
        raise KeyError(f"Kolom wajib tidak ada: {hilang}")
    if KEY_COL not in df.columns:
        df[KEY_COL] = np.arange(len(df))
        logger.warning(f"{KEY_COL} tidak ada, dibuat dari indeks baris.")
    if df[KEY_COL].duplicated().any():
        raise ValueError(f"{KEY_COL} tidak unik. Perbaiki dulu sebelum split.")
    logger.info(f"Dimuat: {len(df)} baris dari {FILE_INPUT.name}")
    return df


def hitung_prioritas(df: pd.DataFrame) -> pd.DataFrame:
    lab_cols = [f"label_{a}" for a in ALIAS_MODEL]
    lab = df[lab_cols].astype(str).apply(lambda s: s.str.strip().str.lower())

    tak_valid = int((~lab.isin(VALID_CLS)).any(axis=1).sum())
    if tak_valid:
        logger.warning(f"{tak_valid} baris memuat label di luar {VALID_CLS}.")

    hitung = pd.DataFrame({c: lab.eq(c).sum(axis=1) for c in VALID_CLS})
    n_setuju = hitung.max(axis=1)
    konsensus = hitung.idxmax(axis=1).where(n_setuju >= 2, "ambigu")

    conf_cols = [f"conf_{a}" for a in ALIAS_MODEL]
    ada_conf = all(c in df.columns for c in conf_cols)
    if ada_conf:
        conf = df[conf_cols].apply(pd.to_numeric, errors="coerce")
        conf.columns = ALIAS_MODEL
        cocok = pd.DataFrame({a: lab[f"label_{a}"].eq(konsensus) for a in ALIAS_MODEL})
        conf_kons = conf.where(cocok).mean(axis=1).fillna(conf.mean(axis=1))
        w_sepakat, w_conf = BOBOT_SEPAKAT, BOBOT_CONF
    else:
        conf_kons = pd.Series(np.nan, index=df.index)
        w_sepakat, w_conf = 1.0, 0.0
        logger.warning(
            "Kolom conf_* tidak lengkap, skor prioritas memakai ketidaksepakatan saja."
        )

    tidak_sepakat = (3 - n_setuju) / 2
    tidak_yakin = (1 - conf_kons).clip(0, 1).fillna(0) if ada_conf else 0.0

    out = df.copy()
    out["label_konsensus"] = konsensus
    out["n_setuju"] = n_setuju
    out["conf_konsensus"] = conf_kons.round(4)
    out["skor_prioritas"] = (w_sepakat * tidak_sepakat + w_conf * tidak_yakin).round(4)
    return out


def susun_keluaran(sub: pd.DataFrame, conf_ada: bool) -> pd.DataFrame:
    sub = sub.copy()
    sub["prioritas_pct"] = (
        sub["skor_prioritas"].rank(pct=True, method="average") * 100
    ).round(2)
    sub["perlu_review"] = sub["prioritas_pct"] >= (100 - AMBANG_PERSENTIL)
    sub["label_manual"] = ""
    sub["catatan"] = ""
    sub = sub.sort_values(
        "skor_prioritas", ascending=False, kind="mergesort"
    ).reset_index(drop=True)

    if SERTAKAN_SEMUA_KOLOM:
        return sub

    kolom = [KEY_COL]
    if DATE_COL in sub.columns:
        kolom.append(DATE_COL)
    kolom += [TEXT_COL, "label_manual", "catatan", "label_konsensus", "n_setuju"]
    kolom += [f"label_{a}" for a in ALIAS_MODEL]
    if conf_ada:
        kolom += ["conf_konsensus"] + [f"conf_{a}" for a in ALIAS_MODEL]
    kolom += ["skor_prioritas", "prioritas_pct", "perlu_review"]
    return sub[[c for c in kolom if c in sub.columns]]


def split() -> None:
    df = hitung_prioritas(muat())
    conf_ada = all(f"conf_{a}" in df.columns for a in ALIAS_MODEL)

    polar = df[df["label_konsensus"].isin(["pos", "neg", "ambigu"])]
    netral = df[df["label_konsensus"].eq("neu")]

    for nama, sub, path in (
        ("POLAR", polar, FILE_POLAR),
        ("NETRAL", netral, FILE_NETRAL),
    ):
        keluar = susun_keluaran(sub, conf_ada)
        keluar.to_csv(path, index=False, encoding=ENC)
        n_rev = int(keluar["perlu_review"].sum())
        logger.info(
            f"[{nama}] {len(keluar)} baris -> {path.name} | "
            f"komposisi={sub['label_konsensus'].value_counts().to_dict()} | "
            f"perlu_review(top {AMBANG_PERSENTIL:.0f}%)={n_rev}"
        )

    print("\nDistribusi konsensus:")
    print(df["label_konsensus"].value_counts().to_string())
    print("\nSebaran kesepakatan (n_setuju):")
    print(df["n_setuju"].value_counts().sort_index().to_string())
    print("\nKuantil skor_prioritas:")
    print(
        df["skor_prioritas"].quantile([0.5, 0.75, 0.9, 0.95, 1.0]).round(4).to_string()
    )
    print(
        "\nLangkah berikutnya: isi kolom label_manual (pos/neu/neg) pada kedua berkas, "
        "biarkan kosong bila label konsensus sudah benar, lalu jalankan MODE='MERGE'."
    )


# ==================================================================
# MERGE
# ==================================================================
def merge() -> None:
    df = hitung_prioritas(muat())
    df = df.drop(
        columns=[
            c
            for c in ("label_manual", "label_final", "sumber_label")
            if c in df.columns
        ]
    )
    bagian = []
    for path in (FILE_POLAR, FILE_NETRAL):
        if not path.exists():
            raise FileNotFoundError(f"{path} belum ada. Jalankan MODE='SPLIT' dulu.")
        rev = pd.read_csv(path, encoding_errors="replace")
        if "label_manual" not in rev.columns:
            raise KeyError(f"{path.name} tidak punya kolom label_manual.")
        rev["label_manual"] = rev["label_manual"].astype(str).str.strip().str.lower()
        bagian.append(
            rev.loc[rev["label_manual"].isin(VALID_CLS), [KEY_COL, "label_manual"]]
        )

    gold = pd.concat(bagian, ignore_index=True).drop_duplicates(
        subset=KEY_COL, keep="last"
    )
    if gold.empty:
        raise ValueError("Belum ada satu pun label_manual yang valid.")

    out = df.merge(gold, on=KEY_COL, how="left")
    out["label_final"] = out["label_manual"].where(
        out["label_manual"].notna(), out["label_konsensus"]
    )
    out["sumber_label"] = np.where(out["label_manual"].notna(), "manual", "konsensus")

    sisa_ambigu = int(out["label_final"].eq("ambigu").sum())
    if sisa_ambigu:
        logger.warning(f"{sisa_ambigu} baris masih 'ambigu' (belum dilabeli manual).")

    out.to_csv(FILE_OPTIMAL, index=False, encoding=ENC)
    logger.info(f"Tersimpan: {FILE_OPTIMAL} ({len(out)} baris)")

    print(f"\nLabel manual terpakai: {len(gold)} baris")
    ubah = int(
        (
            out["sumber_label"].eq("manual")
            & out["label_final"].ne(out["label_konsensus"])
        ).sum()
    )
    print(f"Koreksi terhadap konsensus: {ubah} baris")
    print("\nDistribusi label_final:")
    print(out["label_final"].value_counts().to_string())


# ==================================================================
# LAPORAN: BERKAS KETERANGAN (.md) + VISUALISASI (.png)
# ==================================================================
def _angka(v) -> str:
    if isinstance(v, (bool, np.bool_)):
        return "Ya" if v else "Tidak"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}".replace(",", ".")
    if isinstance(v, (float, np.floating)):
        if np.isnan(v):
            return "-"
        s = f"{v:.4f}" if abs(v) < 1 else f"{v:.2f}"
        return s.replace(".", ",") if DESIMAL_KOMA else s
    return str(v)


def _md(df: pd.DataFrame, judul: str) -> str:
    kol = list(df.columns)
    baris = ["| " + " | ".join(kol) + " |", "|" + "|".join(["---"] * len(kol)) + "|"]
    for _, r in df.iterrows():
        baris.append("| " + " | ".join(_angka(v) for v in r) + " |")
    return f"**{judul}**\n\n" + "\n".join(baris) + "\n"


def _frek(s: pd.Series, nama: str, urutan=None) -> pd.DataFrame:
    n = s.value_counts()
    if urutan is not None:
        n = n.reindex([u for u in urutan if u in n.index]).dropna().astype(int)
    return pd.DataFrame(
        {
            nama: n.index.astype(str),
            "Jumlah": n.values,
            "Persen (%)": (n.values / len(s) * 100),
        }
    )


def laporan() -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update(
        {
            "figure.figsize": (12, 6),
            "font.family": "serif",
            "font.size": 11,
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "savefig.bbox": "tight",
        }
    )
    plt.style.use("seaborn-v0_8-whitegrid")
    FOLDER_VISUAL.mkdir(parents=True, exist_ok=True)

    df = hitung_prioritas(muat())
    n_total = len(df)
    lab_cols = [f"label_{a}" for a in ALIAS_MODEL]
    lab = df[lab_cols].astype(str).apply(lambda s: s.str.strip().str.lower())
    polar = df[df["label_konsensus"].isin(["pos", "neg", "ambigu"])]
    netral = df[df["label_konsensus"].eq("neu")]

    bagian = [
        f"# Laporan Pemisahan Berkas Review Manual\n",
        f"Berkas sumber: `{FILE_INPUT.name}`  \n"
        f"Jumlah baris: {_angka(n_total)}  \n"
        f"Waktu eksekusi: {datetime.now():%Y-%m-%d %H:%M:%S}\n",
        _md(
            pd.DataFrame(
                {
                    "Parameter": [
                        "Acuan kelas",
                        "Bobot ketidaksepakatan",
                        "Bobot ketidakyakinan",
                        "Ambang persentil review",
                        "Model pembanding",
                    ],
                    "Nilai": [
                        "Konsensus mayoritas 3 model",
                        _angka(BOBOT_SEPAKAT),
                        _angka(BOBOT_CONF),
                        f"{_angka(AMBANG_PERSENTIL)}%",
                        ", ".join(ALIAS_MODEL),
                    ],
                }
            ),
            "Tabel 1. Parameter yang digunakan",
        ),
    ]

    bagian.append(
        _md(
            _frek(
                df["label_konsensus"].map(NAMA_KELAS).fillna(df["label_konsensus"]),
                "Kelas konsensus",
                [NAMA_KELAS[k] for k in ("pos", "neu", "neg", "ambigu")],
            ),
            "Tabel 2. Distribusi kelas konsensus",
        )
    )

    peta = {
        3: "Bulat (3 dari 3 model)",
        2: "Mayoritas (2 dari 3 model)",
        1: "Tanpa mayoritas",
    }
    bagian.append(
        _md(
            _frek(df["n_setuju"].map(peta), "Tingkat kesepakatan", list(peta.values())),
            "Tabel 3. Tingkat kesepakatan antarmodel",
        )
    )

    pas = [
        {
            "Pasangan model": f"{a} vs {b}",
            "Jumlah sama": int(lab[f"label_{a}"].eq(lab[f"label_{b}"]).sum()),
            "Persen (%)": lab[f"label_{a}"].eq(lab[f"label_{b}"]).mean() * 100,
        }
        for a, b in combinations(ALIAS_MODEL, 2)
    ]
    bagian.append(_md(pd.DataFrame(pas), "Tabel 4. Kesepakatan berpasangan antarmodel"))

    ringkas = []
    for nama, sub in (("Polar (pos/neg/ambigu)", polar), ("Netral", netral)):
        if sub.empty:
            continue
        amb = float(np.percentile(sub["skor_prioritas"], 100 - AMBANG_PERSENTIL))
        n_rev = int((sub["skor_prioritas"] >= amb).sum())
        ringkas.append(
            {
                "Berkas": nama,
                "Jumlah baris": len(sub),
                "Persen dari total (%)": len(sub) / n_total * 100,
                "Skor prioritas rata-rata": float(sub["skor_prioritas"].mean()),
                f"Ambang persentil {int(AMBANG_PERSENTIL)}%": amb,
                "Baris prioritas review": n_rev,
            }
        )
    bagian.append(_md(pd.DataFrame(ringkas), "Tabel 5. Komposisi berkas keluaran"))

    kuant = pd.DataFrame({"Kuantil": ["50%", "75%", "90%", "95%", "Maksimum"]})
    for nama, sub in (("Polar", polar), ("Netral", netral)):
        if sub.empty:
            continue
        kuant[nama] = (
            sub["skor_prioritas"].quantile([0.5, 0.75, 0.9, 0.95, 1.0]).round(4).values
        )
    bagian.append(_md(kuant, "Tabel 6. Kuantil skor prioritas"))

    if not netral.empty:
        idx = netral.index
        ada_polar = lab.loc[idx].isin(["pos", "neg"]).any(axis=1)
        kat = np.where(
            ada_polar,
            "Ada model menilai positif/negatif",
            "Seluruh model menilai netral",
        )
        bagian.append(
            _md(
                _frek(
                    pd.Series(kat, index=idx),
                    "Kategori",
                    [
                        "Ada model menilai positif/negatif",
                        "Seluruh model menilai netral",
                    ],
                ),
                "Tabel 7. Potensi kesalahan label pada berkas netral",
            )
        )

    if FILE_OPTIMAL.exists():
        opt = pd.read_csv(FILE_OPTIMAL, encoding_errors="replace")
        if {"label_final", "label_konsensus"}.issubset(opt.columns):
            a = opt["label_konsensus"].value_counts()
            b = opt["label_final"].value_counts()
            hadir = set(a.index) | set(b.index)
            kelas = [k for k in ("pos", "neu", "neg", "ambigu") if k in hadir]
            kelas += sorted(hadir - set(kelas))
            band = pd.DataFrame({"Kelas": [NAMA_KELAS.get(k, k) for k in kelas]})
            band["Konsensus"] = [int(a.get(k, 0)) for k in kelas]
            band["Setelah review"] = [int(b.get(k, 0)) for k in kelas]
            band["Selisih"] = band["Setelah review"] - band["Konsensus"]
            band["Perubahan (%)"] = np.where(
                band["Konsensus"] > 0, band["Selisih"] / band["Konsensus"] * 100, np.nan
            )
            bagian.append(
                _md(band, "Tabel 8. Perbandingan sebelum dan sesudah review manual")
            )
            n_ubah = int(opt["label_final"].ne(opt["label_konsensus"]).sum())
            bagian.append(
                f"Koreksi manual: {_angka(n_ubah)} baris "
                f"({_angka(n_ubah / len(opt) * 100)}% dari total).\n"
            )

    FILE_LAPORAN.write_text("\n".join(bagian), encoding="utf-8")
    logger.info(f"Tersimpan: {FILE_LAPORAN}")

    def _pct(v: float) -> str:
        s = f"{v:.1f}%"
        return s.replace(".", ",") if DESIMAL_KOMA else s

    urut = [
        k for k in ("pos", "neu", "neg", "ambigu") if k in df["label_konsensus"].values
    ]
    n_kelas = df["label_konsensus"].value_counts().reindex(urut)
    fig, ax = plt.subplots()
    ax.pie(
        n_kelas.values,
        labels=[f"{NAMA_KELAS[k]}\n(n={_angka(int(n_kelas[k]))})" for k in urut],
        colors=[WARNA[k] for k in urut],
        autopct=_pct,
        startangle=140,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    )
    ax.set_title("Distribusi Kelas Konsensus Tiga Model", fontweight="bold")
    f1 = FOLDER_VISUAL / "1_distribusi_konsensus.png"
    fig.savefig(f1, dpi=DPI)
    plt.close(fig)

    fig, ax = plt.subplots()
    for nama, sub, warna in (
        ("Polar", polar, "#2980b9"),
        ("Netral", netral, "#7f8c8d"),
    ):
        if sub.empty:
            continue
        s = sub["skor_prioritas"].to_numpy()
        ax.hist(
            s,
            bins=BIN_HISTOGRAM,
            weights=np.ones(len(s)) / len(s) * 100,
            alpha=0.55,
            color=warna,
            label=f"{nama} (n={_angka(len(s))})",
        )
        amb = float(np.percentile(s, 100 - AMBANG_PERSENTIL))
        teks = f"{amb:.3f}".replace(".", ",") if DESIMAL_KOMA else f"{amb:.3f}"
        ax.axvline(
            amb,
            color=warna,
            linestyle="--",
            linewidth=1.8,
            label=f"Ambang {nama} = {teks}",
        )
    ax.set_title(
        f"Sebaran Skor Prioritas Review Manual (ambang {AMBANG_PERSENTIL:.0f}% teratas)",
        fontweight="bold",
    )
    ax.set_xlabel("Skor prioritas (0 = paling yakin, 1 = paling meragukan)")
    ax.set_ylabel("Persentase baris dalam berkas (%)")
    if DESIMAL_KOMA:
        ax.xaxis.set_major_formatter(
            plt.FuncFormatter(lambda v, _: f"{v:.2f}".replace(".", ","))
        )
    ax.legend(frameon=True, shadow=True)
    f2 = FOLDER_VISUAL / "2_histogram_prioritas.png"
    fig.savefig(f2, dpi=DPI)
    plt.close(fig)

    logger.info(f"Tersimpan: {f1.name}, {f2.name} di {FOLDER_VISUAL}")
    print(f"\nLaporan: {FILE_LAPORAN}\nVisualisasi: {FOLDER_VISUAL}")


# ==================================================================
# ENTRY POINT
# ==================================================================
if __name__ == "__main__":
    FOLDER_OUTPUT.mkdir(parents=True, exist_ok=True)
    if MODE == "SPLIT":
        split()
    elif MODE == "MERGE":
        merge()
    elif MODE == "LAPORAN":
        laporan()
    else:
        raise ValueError("MODE harus 'SPLIT', 'MERGE', atau 'LAPORAN'.")

2026-08-16 15:37:04,945 - INFO - Dimuat: 14050 baris dari dataset_multilabel_final.csv
2026-08-16 15:37:05,067 - INFO - Tersimpan: G:\My Drive\skripsi\proje\proje\data\hasil\laporan_split_review.md
2026-08-16 15:37:05,834 - INFO - Tersimpan: 1_distribusi_konsensus.png, 2_histogram_prioritas.png di G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi



Laporan: G:\My Drive\skripsi\proje\proje\data\hasil\laporan_split_review.md
Visualisasi: G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi


Gambaran visualisasi terakhir, gabungan setelah perubahan sentimen netral


In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

DIR_OUT = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi")
FILE_INPUT = DIR_OUT / "dataset_final_gabungan.csv"

KOL_LABEL = "label_final"
DATE_COL = "tanggal_wib"
KELAS = ["neg", "neu", "pos"]
NAMA = {"neg": "Negatif", "neu": "Netral", "pos": "Positif"}
WARNA = {"neg": "#c0392b", "neu": "#95a5a6", "pos": "#27ae60"}
FREKUENSI = "D"


def main():
    if not FILE_INPUT.exists():
        ada = sorted(p.name for p in DIR_OUT.glob("*.csv")) if DIR_OUT.exists() else []
        raise SystemExit(f"Tidak ditemukan: {FILE_INPUT}\nCSV di folder itu: {ada}")
    df = pd.read_csv(FILE_INPUT)
    for c in (KOL_LABEL, DATE_COL):
        if c not in df.columns:
            raise SystemExit(f"Kolom '{c}' tidak ada: {df.columns.tolist()}")

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    n_rusak = int(df[DATE_COL].isna().sum())
    if n_rusak:
        print(f"PERINGATAN: {n_rusak} baris tanggalnya tidak terbaca, dibuang")
    df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
    df["tanggal"] = df[DATE_COL].dt.date

    n = len(df)
    print(
        f"{FILE_INPUT.name}: {n} baris, {df[DATE_COL].min():%Y-%m-%d} s.d. {df[DATE_COL].max():%Y-%m-%d}"
    )

    jml = df[KOL_LABEL].value_counts().reindex(KELAS).fillna(0).astype(int)
    pct = (jml / n * 100).round(2)
    total = pd.DataFrame(
        {"kelas": [NAMA[k] for k in KELAS], "jumlah": jml.values, "persen": pct.values}
    )
    polar = jml["pos"] + jml["neg"]
    print("\nPERSENTASE TOTAL")
    for k in KELAS:
        print(f"  {NAMA[k]:<8} {jml[k]:6d}  {pct[k]:6.2f}%")
    print(f"  {'Total':<8} {n:6d}  100.00%")
    print(f"\nDi antara polar saja (tanpa netral, n={polar}):")
    print(
        f"  Positif {jml['pos'] / polar * 100:6.2f}%  |  Negatif {jml['neg'] / polar * 100:6.2f}%"
    )
    print(f"NSt keseluruhan: {(jml['pos'] - jml['neg']) / n:.4f}")

    harian = (
        df.groupby("tanggal")[KOL_LABEL]
        .value_counts()
        .unstack(fill_value=0)
        .reindex(columns=KELAS, fill_value=0)
        .reset_index()
        .sort_values("tanggal")
    )
    harian["n_total"] = harian[KELAS].sum(axis=1)
    for k in KELAS:
        harian[f"pct_{k}"] = (harian[k] / harian["n_total"] * 100).round(2)
    harian["NSt"] = ((harian["pos"] - harian["neg"]) / harian["n_total"]).round(6)
    print(
        f"\nAgregasi harian: {len(harian)} hari, NSt rata-rata {harian['NSt'].mean():.4f}, "
        f"min {harian['NSt'].min():.4f}, maks {harian['NSt'].max():.4f}"
    )

    DIR_OUT.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        DIR_OUT / "dataset_final_urut_waktu.csv", index=False, encoding="utf-8-sig"
    )
    total.to_csv(DIR_OUT / "persentase_total.csv", index=False, encoding="utf-8-sig")
    harian.to_csv(DIR_OUT / "persentase_harian.csv", index=False, encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.barh(
        [NAMA[k] for k in KELAS],
        [pct[k] for k in KELAS],
        color=[WARNA[k] for k in KELAS],
    )
    for i, k in enumerate(KELAS):
        ax.text(
            pct[k] + 0.6, i, f"{pct[k]:.2f}%  (n={jml[k]:,})", va="center", fontsize=9
        )
    ax.set_xlim(0, max(pct) * 1.25)
    ax.set_xlabel("Persentase terhadap total tweet (%)")
    ax.set_title(f"Komposisi Sentimen Keseluruhan (n = {n:,})")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(DIR_OUT / "grafik_persentase_total.png", dpi=200)
    plt.close(fig)

    idx = pd.to_datetime(harian["tanggal"])
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.stackplot(
        idx,
        *[harian[f"pct_{k}"] for k in KELAS],
        labels=[NAMA[k] for k in KELAS],
        colors=[WARNA[k] for k in KELAS],
    )
    ax.set_ylim(0, 100)
    ax.set_xlim(idx.min(), idx.max())
    ax.set_ylabel("Komposisi harian (%)")
    ax.set_title("Komposisi Sentimen Harian")
    ax.legend(loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.12))
    fig.tight_layout()
    fig.savefig(DIR_OUT / "grafik_komposisi_harian.png", dpi=200)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.axhline(0, color="#7f8c8d", lw=1)
    ax.plot(idx, harian["NSt"], color="#2c3e50", lw=1.2)
    ax.fill_between(
        idx, harian["NSt"], 0, where=harian["NSt"] >= 0, color=WARNA["pos"], alpha=0.35
    )
    ax.fill_between(
        idx, harian["NSt"], 0, where=harian["NSt"] < 0, color=WARNA["neg"], alpha=0.35
    )
    ax.set_xlim(idx.min(), idx.max())
    ax.set_ylabel("NSt")
    ax.set_title("Net Sentiment (NSt) Harian")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(DIR_OUT / "grafik_nst_harian.png", dpi=200)
    plt.close(fig)

    for f in (
        "dataset_final_urut_waktu.csv",
        "persentase_total.csv",
        "persentase_harian.csv",
        "grafik_persentase_total.png",
        "grafik_komposisi_harian.png",
        "grafik_nst_harian.png",
    ):
        print(f"Tersimpan: {f}")


if __name__ == "__main__":
    main()

dataset_final_gabungan.csv: 14050 baris, 2025-06-01 s.d. 2025-10-31

PERSENTASE TOTAL
  Negatif    5398   38.42%
  Netral     3449   24.55%
  Positif    4894   34.83%
  Total     14050  100.00%

Di antara polar saja (tanpa netral, n=10292):
  Positif  47.55%  |  Negatif  52.45%
NSt keseluruhan: -0.0359

Agregasi harian: 153 hari, NSt rata-rata -0.0201, min -0.6250, maks 1.0000
Tersimpan: dataset_final_urut_waktu.csv
Tersimpan: persentase_total.csv
Tersimpan: persentase_harian.csv
Tersimpan: grafik_persentase_total.png
Tersimpan: grafik_komposisi_harian.png
Tersimpan: grafik_nst_harian.png


In [ ]:
import pandas as pd
from pathlib import Path

FILE_IN = Path(
    r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\dataset_final_urut_waktu.csv"
)
FILE_OUT = FILE_IN.parent / "dataset_final_rk2_lengkap.csv"

df = pd.read_csv(FILE_IN, encoding="utf-8-sig")
kosong = df["label_final"].isna()
print(f"label_final kosong sebelum: {kosong.sum()}")

# Verifikasi: apakah semua n_setuju == 1?
if "n_setuju" in df.columns:
    print("n_setuju pada baris kosong:")
    print(df.loc[kosong, "n_setuju"].value_counts().to_string())

# Kolom confidence dan label tiap model
MODEL_MAP = {
    "mdhugol": ("label_mdhugol", "conf_mdhugol"),
    "w11wo_roberta": ("label_w11wo_roberta", "conf_w11wo_roberta"),
    "ayame_bertcahya": ("label_ayame_bertcahya", "conf_ayame_bertcahya"),
}

# Untuk tiap baris kosong: ambil label dari model dengan conf tertinggi
for idx in df.index[kosong]:
    best_model, best_conf, best_label = None, -1.0, None
    for alias, (col_lab, col_conf) in MODEL_MAP.items():
        c = df.at[idx, col_conf]
        if pd.notna(c) and c > best_conf:
            best_conf = c
            best_label = df.at[idx, col_lab]
            best_model = alias
    if best_label is not None:
        df.at[idx, "label_final"] = best_label
        df.at[idx, "sumber_label_final"] = f"seri_confidence_{best_model}"

sisa = int(df["label_final"].isna().sum())
print(f"label_final kosong sesudah: {sisa}")
print(f"\nDistribusi label_final:")
print(df["label_final"].value_counts(dropna=False).to_string())
print(f"\nDistribusi sumber_label_final:")
print(df["sumber_label_final"].value_counts(dropna=False).to_string())

df.to_csv(FILE_OUT, index=False, encoding="utf-8-sig")
print(f"\nDisimpan: {FILE_OUT.name}")

label_final kosong sebelum: 309
n_setuju pada baris kosong:
n_setuju
1    309
label_final kosong sesudah: 0

Distribusi label_final:
label_final
neg    5506
pos    4957
neu    3587

Distribusi sumber_label_final:
sumber_label_final
konsensus                          9294
leksikon                           4447
seri_confidence_ayame_bertcahya     205
seri_confidence_mdhugol              55
seri_confidence_w11wo_roberta        49

Disimpan: dataset_final_rk2_lengkap.csv


coba lihat ini!!!!!

In [ ]:
import pandas as pd

df = pd.read_csv(
    r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\dataset_final_rk2_lengkap.csv"
)

# 1. Cek Kekosongan
kosong = df["label_final"].isna().sum()
print(
    f"Data kosong: {kosong} baris. {'(Perbaikan: Jalankan ulang script fallback)' if kosong > 0 else '(Aman)'}\n"
)

# 2. Riwayat Data yang Diubah (Diisi Otomatis)
diubah = df[df["sumber_label_final"].str.contains("seri_confidence", na=False)]
print(f"Total data diubah otomatis: {len(diubah)} baris")
print(diubah["sumber_label_final"].value_counts().to_string(), "\n")

# 3. Persentase Final
c = df["label_final"].value_counts()
print("Persentase Total:")
for lbl in ["neg", "neu", "pos"]:
    print(
        f"  {lbl.upper():<7}: {c.get(lbl, 0):<5} ({c.get(lbl, 0) / len(df) * 100:.2f}%)"
    )

polar = c.get("pos", 0) + c.get("neg", 0)
if polar > 0:
    print(f"\nPolar saja (tanpa netral, n={polar}):")
    print(
        f"  POS: {c.get('pos', 0) / polar * 100:.2f}% | NEG: {c.get('neg', 0) / polar * 100:.2f}%"
    )

Data kosong: 0 baris. (Aman)

Total data diubah otomatis: 309 baris
sumber_label_final
seri_confidence_ayame_bertcahya    205
seri_confidence_mdhugol             55
seri_confidence_w11wo_roberta       49 

Persentase Total:
  NEG    : 5506  (39.19%)
  NEU    : 3587  (25.53%)
  POS    : 4957  (35.28%)

Polar saja (tanpa netral, n=10463):
  POS: 47.38% | NEG: 52.62%


In [ ]:
"""
P5: Validasi Konsensus Pelabelan (tanpa gold sample eksternal)
=============================================================
Tujuan : Memvalidasi kualitas label_final dengan mengukur
         kesepakatan (agreement) antar model IndoBERT.
         Tidak memerlukan file gold sample terpisah.

Input  : dataset_final_rk2_lengkap.csv
Output : laporan di terminal + laporan_konsensus_p5.csv

"""

import pandas as pd
import numpy as np
from pathlib import Path

# ── KONFIGURASI ──────────────────────────────────────────────────────
FILE_IN = Path(
    r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\dataset_final_rk2_lengkap.csv"
)
FILE_OUT = FILE_IN.parent / "laporan_konsensus_p5.csv"

TEXT_COL = "text_clean_bert"
VALID = ["neg", "neu", "pos"]

MODEL_MAP = {
    "mdhugol": ("label_mdhugol", "conf_mdhugol"),
    "w11wo_roberta": ("label_w11wo_roberta", "conf_w11wo_roberta"),
    "ayame_bertcahya": ("label_ayame_bertcahya", "conf_ayame_bertcahya"),
}
alias_list = list(MODEL_MAP)
lab_cols = [MODEL_MAP[a][0] for a in alias_list]
conf_cols = [MODEL_MAP[a][1] for a in alias_list]

# =====================================================================
# 1. BACA DATA
# =====================================================================
print("=" * 68)
print("P5: VALIDASI KONSENSUS PELABELAN")
print("=" * 68)

df = pd.read_csv(FILE_IN, encoding="utf-8-sig", low_memory=False)
df.columns = [c.lstrip("\ufeff").strip() for c in df.columns]
print(f"\nFile  : {FILE_IN.name}")
print(f"Baris : {len(df)}")

# Normalisasi label model
for c in lab_cols:
    df[c] = df[c].astype(str).str.strip().str.lower()
    df[c] = df[c].where(df[c].isin(VALID), np.nan)

# =====================================================================
# 2. DISTRIBUSI LABEL FINAL
# =====================================================================
print("\n" + "-" * 68)
print("DISTRIBUSI LABEL_FINAL")
print("-" * 68)

if "label_final" not in df.columns:
    raise ValueError("Kolom label_final tidak ditemukan. Jalankan P2 terlebih dahulu.")

df["label_final"] = df["label_final"].astype(str).str.strip().str.lower()
df["label_final"] = df["label_final"].where(df["label_final"].isin(VALID), np.nan)

n_terisi = int(df["label_final"].notna().sum())
n_kosong = int(df["label_final"].isna().sum())
print(f"  Terisi : {n_terisi}/{len(df)} ({n_terisi / len(df) * 100:.1f}%)")
print(f"  Kosong : {n_kosong}")

vc = df["label_final"].value_counts(dropna=False)
for val, cnt in vc.items():
    label = str(val) if pd.notna(val) else "(NaN)"
    print(f"    {label:8s}: {cnt:6d} ({cnt / len(df) * 100:.2f}%)")

# =====================================================================
# 3. KESEPAKATAN ANTAR MODEL (INTER-MODEL AGREEMENT)
# =====================================================================
print("\n" + "-" * 68)
print("KESEPAKATAN ANTAR MODEL (Inter-Model Agreement)")
print("-" * 68)

model_labels = df[lab_cols].copy()
n_model_valid = model_labels.notna().sum(axis=1)

vote_counts = pd.DataFrame(
    {k: (model_labels == k).sum(axis=1) for k in VALID}, index=df.index
)
n_setuju = vote_counts.max(axis=1)
label_mayoritas = vote_counts.idxmax(axis=1)
is_unanimous = n_setuju == 3

print(
    f"\n  Ketiga model sepakat (3/3) : {int(is_unanimous.sum()):6d} ({is_unanimous.mean() * 100:.1f}%)"
)
print(
    f"  Mayoritas 2 vs 1           : {int((n_setuju == 2).sum()):6d} ({(n_setuju == 2).mean() * 100:.1f}%)"
)
print(
    f"  Tidak ada mayoritas (seri)  : {int((n_setuju == 1).sum()):6d} ({(n_setuju == 1).mean() * 100:.1f}%)"
)

# =====================================================================
# 4. AGREEMENT PAIRWISE ANTAR MODEL
# =====================================================================
print("\n" + "-" * 68)
print("AGREEMENT PAIRWISE ANTAR MODEL")
print("-" * 68)

pair_data = []
for i in range(len(alias_list)):
    for j in range(i + 1, len(alias_list)):
        a1, a2 = alias_list[i], alias_list[j]
        c1, c2 = lab_cols[i], lab_cols[j]
        mask = df[c1].notna() & df[c2].notna()
        if mask.sum() > 0:
            agree = (df.loc[mask, c1] == df.loc[mask, c2]).mean() * 100
            pair_data.append(
                {
                    "pasangan": f"{a1} vs {a2}",
                    "n_banding": int(mask.sum()),
                    "agreement_%": round(agree, 1),
                }
            )
            print(f"  {a1:20s} vs {a2:20s}: {agree:.1f}% (n={int(mask.sum())})")

# =====================================================================
# 5. KONSISTENSI LABEL_FINAL DENGAN MAYORITAS
# =====================================================================
print("\n" + "-" * 68)
print("KONSISTENSI LABEL_FINAL vs MAJORITY VOTE")
print("-" * 68)

has_majority = n_setuju >= 2
label_maj = label_mayoritas.where(has_majority, np.nan)

mask_both = df["label_final"].notna() & label_maj.notna()
if mask_both.sum() > 0:
    konsisten = (df.loc[mask_both, "label_final"] == label_maj[mask_both]).mean() * 100
    n_beda = int((df.loc[mask_both, "label_final"] != label_maj[mask_both]).sum())
    print(f"  Baris bisa dibandingkan : {int(mask_both.sum())}")
    print(f"  label_final = mayoritas : {konsisten:.1f}%")
    print(f"  label_final ≠ mayoritas : {n_beda} baris")

    if n_beda > 0 and n_beda <= 20:
        print(f"\n  Detail baris yang berbeda (maks 20):")
        idx_beda = df.index[mask_both & (df["label_final"] != label_maj)][:20]
        for i in idx_beda:
            lf = df.at[i, "label_final"]
            lm = label_maj[i]
            models = " | ".join(
                f"{a}={df.at[i, c]}" for a, c in zip(alias_list, lab_cols)
            )
            print(f"    baris {i}: final={lf}, mayoritas={lm} [{models}]")
else:
    print("  Tidak ada baris yang bisa dibandingkan.")

# =====================================================================
# 6. CONFIDENCE RATA-RATA PER KELAS
# =====================================================================
print("\n" + "-" * 68)
print("CONFIDENCE RATA-RATA PER MODEL & KELAS")
print("-" * 68)

for a, (cl, cc) in MODEL_MAP.items():
    if cc in df.columns:
        print(f"\n  {a}:")
        for kelas in VALID:
            mask_k = df[cl] == kelas
            if mask_k.sum() > 0:
                avg_c = df.loc[mask_k, cc].mean()
                print(
                    f"    {kelas}: conf rata-rata = {avg_c:.4f} (n={int(mask_k.sum())})"
                )

# =====================================================================
# 7. SUMBER LABEL FINAL
# =====================================================================
if "sumber_label_final" in df.columns:
    print("\n" + "-" * 68)
    print("DISTRIBUSI SUMBER LABEL_FINAL")
    print("-" * 68)
    for val, cnt in df["sumber_label_final"].value_counts(dropna=False).items():
        label = str(val) if pd.notna(val) else "(NaN)"
        pct = cnt / len(df) * 100
        print(f"  {label:35s}: {cnt:6d} ({pct:.1f}%)")

# =====================================================================
# 8. SIMPAN LAPORAN
# =====================================================================
print("\n" + "-" * 68)
print("SIMPAN LAPORAN")
print("-" * 68)

ringkasan = {
    "total_baris": len(df),
    "label_final_terisi": n_terisi,
    "label_final_kosong": n_kosong,
    "sepakat_3_model": int(is_unanimous.sum()),
    "sepakat_3_model_pct": round(is_unanimous.mean() * 100, 1),
    "mayoritas_2v1": int((n_setuju == 2).sum()),
    "mayoritas_2v1_pct": round((n_setuju == 2).mean() * 100, 1),
    "seri_tanpa_mayoritas": int((n_setuju == 1).sum()),
    "seri_tanpa_mayoritas_pct": round((n_setuju == 1).mean() * 100, 1),
}

if mask_both.sum() > 0:
    ringkasan["final_vs_mayoritas_konsisten_pct"] = round(konsisten, 1)
    ringkasan["final_vs_mayoritas_beda"] = n_beda

for p in pair_data:
    key = p["pasangan"].replace(" vs ", "_vs_").replace(" ", "_")
    ringkasan[f"agreement_{key}"] = p["agreement_%"]

laporan = pd.DataFrame([ringkasan])
laporan.to_csv(FILE_OUT, index=False, encoding="utf-8-sig")
print(f"  → Disimpan: {FILE_OUT.name}")

# =====================================================================
# 9. KEPUTUSAN P5
# =====================================================================
print("\n" + "=" * 68)
print("KEPUTUSAN P5")
print("=" * 68)

overall_agreement = is_unanimous.mean() * 100
if overall_agreement >= 80:
    status = "SANGAT BAIK"
elif overall_agreement >= 60:
    status = "BAIK"
elif overall_agreement >= 40:
    status = "CUKUP"
else:
    status = "RENDAH — pertimbangkan review manual"

print(f"""
  Kesepakatan 3 model : {overall_agreement:.1f}% → {status}
  Pelabelan dilakukan dengan 3 model IndoBERT + majority voting.

  Final : {"LOLOS ✓" if overall_agreement >= 50 else "PERLU REVIEW ✗"}
""")

P5: VALIDASI KONSENSUS PELABELAN

File  : dataset_final_rk2_lengkap.csv
Baris : 14050

--------------------------------------------------------------------
DISTRIBUSI LABEL_FINAL
--------------------------------------------------------------------
  Terisi : 14050/14050 (100.0%)
  Kosong : 0
    neg     :   5506 (39.19%)
    pos     :   4957 (35.28%)
    neu     :   3587 (25.53%)

--------------------------------------------------------------------
KESEPAKATAN ANTAR MODEL (Inter-Model Agreement)
--------------------------------------------------------------------

  Ketiga model sepakat (3/3) :   9597 (68.3%)
  Mayoritas 2 vs 1           :   4144 (29.5%)
  Tidak ada mayoritas (seri)  :    309 (2.2%)

--------------------------------------------------------------------
AGREEMENT PAIRWISE ANTAR MODEL
--------------------------------------------------------------------
  mdhugol              vs w11wo_roberta       : 77.9% (n=14050)
  mdhugol              vs ayame_bertcahya     : 80.5% (n=

In [ ]:
import pandas as pd
from pathlib import Path

# === KONFIGURASI ===
SRC = Path(
    r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\dataset_final_urut_waktu.csv"
)
OUT = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\FINAL.csv")
KOL_MANUAL = "label_manual"
KOL_MODEL = ["label_mdhugol", "label_w11wo_roberta", "label_ayame_bertcahya"]
VALID = {"pos", "neu", "neg"}

# === 1. BACA DATA ===
df = pd.read_csv(SRC, low_memory=False)
print("Total baris:", len(df))

# === 2. CEK NILAI ASLI ===
print("\n========== NILAI ASLI ==========")
for c in [KOL_MANUAL] + KOL_MODEL:
    print(f"\n{c}:")
    print(df[c].value_counts(dropna=False).head(10))

# === 3. MAPPING OTOMATIS ===
MAPPING = {
    "pos": "pos",
    "neg": "neg",
    "neu": "neu",
    "positive": "pos",
    "negative": "neg",
    "neutral": "neu",
    "positif": "pos",
    "negatif": "neg",
    "netral": "neu",
    "label_0": "pos",
    "label_1": "neu",
    "label_2": "neg",
    "0": "neg",
    "1": "neu",
    "2": "pos",
    "true": "pos",
    "false": "neg",
}

# Normalisasi manual
df[KOL_MANUAL] = df[KOL_MANUAL].astype(str).str.strip().str.lower().map(MAPPING)

# Normalisasi 3 model
for c in KOL_MODEL:
    df[c] = df[c].astype(str).str.strip().str.lower().map(MAPPING)

# === 4. CEK SETELAH MAPPING ===
print("\n========== SETELAH MAPPING ==========")
for c in [KOL_MANUAL] + KOL_MODEL:
    print(f"\n{c}:")
    print(df[c].value_counts(dropna=False).head(10))

# === 5. PISAHKAN: MANUAL vs NON-MANUAL ===
punya_manual = df[KOL_MANUAL].isin(VALID)

print(f"\n========== PEMBAGIAN DATA ==========")
print(f"Punya label manual (langsung valid) : {punya_manual.sum()}")
print(f"Tanpa label manual (perlu voting)   : {(~punya_manual).sum()}")

# === 6. LABEL FINAL ===
# Prioritas 1: Kalau ada label manual → langsung pakai
# Prioritas 2: Kalau tidak ada → voting dari 3 model
df["label_final"] = None

# Baris yang punya manual → langsung valid
df.loc[punya_manual, "label_final"] = df.loc[punya_manual, KOL_MANUAL]
df.loc[punya_manual, "sumber_label"] = "manual"

# Baris tanpa manual → voting 3 model
df.loc[~punya_manual, "label_final"] = df.loc[~punya_manual, KOL_MODEL].mode(axis=1)[0]
df.loc[~punya_manual, "sumber_label"] = "voting_3model"

# === 7. RINGKASAN HASIL ===
print(f"\n========== LABEL FINAL ==========")
print(df["label_final"].value_counts(dropna=False))

print(f"\n========== SUMBER LABEL ==========")
print(df["sumber_label"].value_counts())

print(f"\n========== DISTRIBUSI PER SUMBER ==========")
print(df.groupby("sumber_label")["label_final"].value_counts())

# === 8. SIMPAN ===
df.to_csv(OUT, index=False)
print(f"\nTersimpan → {OUT}")

Total baris: 14050

========== NILAI ASLI ==========

label_manual:
label_manual
NaN    9603
pos    2782
neg    1665
Name: count, dtype: int64

label_mdhugol:
label_mdhugol
neu    8126
neg    3734
pos    2190
Name: count, dtype: int64

label_w11wo_roberta:
label_w11wo_roberta
neu    7026
neg    4100
pos    2924
Name: count, dtype: int64

label_ayame_bertcahya:
label_ayame_bertcahya
neu    8371
neg    3757
pos    1922
Name: count, dtype: int64

========== SETELAH MAPPING ==========

label_manual:
label_manual
NaN    9603
pos    2782
neg    1665
Name: count, dtype: int64

label_mdhugol:
label_mdhugol
neu    8126
neg    3734
pos    2190
Name: count, dtype: int64

label_w11wo_roberta:
label_w11wo_roberta
neu    7026
neg    4100
pos    2924
Name: count, dtype: int64

label_ayame_bertcahya:
label_ayame_bertcahya
neu    8371
neg    3757
pos    1922
Name: count, dtype: int64

========== PEMBAGIAN DATA ==========
Punya label manual (langsung valid) : 4447
Tanpa label manual (perlu voting)   : 9

In [9]:
import pandas as pd
from pathlib import Path

# === KONFIGURASI ===
SRC = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\FINAL.csv")
OUT = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\FINAL 2.csv")
KOL_MANUAL = "label_manual"
KOL_MODEL = ["label_mdhugol", "label_w11wo_roberta", "label_ayame_bertcahya"]
VALID = {"pos", "neu", "neg"}

# === 1. BACA ===
df = pd.read_csv(SRC, low_memory=False)

# === 2. MAPPING ===
MAPPING = {
    "pos": "pos",
    "neg": "neg",
    "neu": "neu",
    "positive": "pos",
    "negative": "neg",
    "neutral": "neu",
    "positif": "pos",
    "negatif": "neg",
    "netral": "neu",
    "label_0": "pos",
    "label_1": "neu",
    "label_2": "neg",
    "0": "neg",
    "1": "neu",
    "2": "pos",
    "true": "pos",
    "false": "neg",
}

for c in [KOL_MANUAL] + KOL_MODEL:
    df[c] = df[c].astype(str).str.strip().str.lower().map(MAPPING)

# === 3. LABEL FINAL ===
punya_manual = df[KOL_MANUAL].isin(VALID)

df["label_final"] = None
df.loc[punya_manual, "label_final"] = df.loc[punya_manual, KOL_MANUAL]
df.loc[~punya_manual, "label_final"] = df.loc[~punya_manual, KOL_MODEL].mode(axis=1)[0]
df["sumber"] = punya_manual.map({True: "manual", False: "voting_3model"})

# === 4. RINGKASAN ===
print("Total:", len(df))
print(f"  Manual : {punya_manual.sum()}")
print(f"  Voting : {(~punya_manual).sum()}")
print(f"\nDistribusi label_final:")
print(df["label_final"].value_counts(dropna=False))

# === 5. BUANG KOLOM LABEL LAMA, SIMPAN BERSIH ===
kolom_buang = [KOL_MANUAL] + KOL_MODEL
df_final = df.drop(columns=kolom_buang)

print(f"\nKolom tersisa: {list(df_final.columns)}")
df_final.to_csv(OUT, index=False)
print(f"Tersimpan → {OUT}")

Total: 14050
  Manual : 4447
  Voting : 9603

Distribusi label_final:
label_final
neg    5707
pos    4894
neu    3449
Name: count, dtype: int64

Kolom tersisa: ['row_key', 'tanggal_wib', 'text_clean_bert', 'catatan', 'label_konsensus', 'n_setuju', 'conf_konsensus', 'conf_mdhugol', 'conf_w11wo_roberta', 'conf_ayame_bertcahya', 'skor_prioritas', 'prioritas_pct', 'perlu_review', 'asal_file', 'n_pos_leks', 'n_neg_leks', 'entri_pemicu', 'label_final', 'sumber_label_final', 'label_berubah', 'tanggal', 'sumber_label', 'sumber']
Tersimpan → G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\FINAL 2.csv


Jika semua diproses untuk sentimen manual : Final 3

In [ ]:
import hashlib
import re
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path(r"C:\Users\EZRA\Downloads\tempat\FINAL 2.csv")
SALINAN_LAIN = Path(
    r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\FINAL 2.csv"
)
DIR_KAMUS = Path(r"C:\Users\EZRA\Downloads\tempat\tes ALL IN")
KAMUS = {
    "frasa": (DIR_KAMUS / "label_frasa_2_5_kata.csv", "teks"),
    "korpus_lanjutan": (DIR_KAMUS / "rekomendasi_label_korpus_lanjutan.csv", "Kata"),
}
KOL_STATUS = "Status Perubahan"

DIR = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi")
DIR_TES = DIR / "tes ALL IN"
OUT = DIR / "FINAL 3.csv"
OUT_FREK = DIR_TES / "frekuensi_entri_kamus.csv"
OUT_JUMLAH = DIR_TES / "jumlah_sentimen_FINAL3.csv"
OUT_PERIODE = DIR_TES / "perubahan_per_periode_FINAL3.csv"
OUT_BERUBAH = DIR_TES / "dokumen_berubah_FINAL2_ke_FINAL3.csv"

IMBANG = "neg"

KEY, TEKS, TGL = "row_key", "text_clean_bert", "tanggal_wib"
PETA = {"positif": "pos", "negatif": "neg", "netral": "neu"}
URUT = ["pos", "neu", "neg"]
NAMA = {"pos": "Positif", "neu": "Netral", "neg": "Negatif"}
SKOR = {"pos": 1, "neu": 0, "neg": -1}
PERIODE = [
    ("Pra-peristiwa", "2025-06-01", "2025-08-24"),
    ("Puncak peristiwa", "2025-08-25", "2025-09-09"),
    ("Pasca-peristiwa", "2025-09-10", "2025-10-31"),
]


def rapikan(s):
    return s.astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()


def sidik(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()


def cek_awal():
    if IMBANG not in URUT + ["lama"]:
        raise SystemExit(f"IMBANG harus salah satu dari {URUT + ['lama']}")
    berkas = [SRC] + [p for p, _ in KAMUS.values()]
    hilang = [str(p) for p in berkas if not p.is_file()]
    if hilang:
        raise SystemExit("Berkas tidak ditemukan:\n" + "\n".join(hilang))
    kol = pd.read_csv(SRC, nrows=0, encoding="utf-8-sig").columns
    kurang = [c for c in (KEY, TGL, TEKS, "label_final", "sumber") if c not in kol]
    if kurang:
        raise SystemExit(f"Kolom {kurang} tidak ada di {SRC.name}")
    if SALINAN_LAIN.is_file() and sidik(SALINAN_LAIN) != sidik(SRC):
        print(
            f"PERINGATAN: {SRC.name} di {SRC.parent} dan {SALINAN_LAIN.parent} berbeda."
        )
        print(f"            Kode ini memakai {SRC}\n")
    DIR_TES.mkdir(parents=True, exist_ok=True)


def muat_kamus():
    bag = []
    for asal, (path, kol) in KAMUS.items():
        k = pd.read_csv(path, encoding="utf-8-sig")
        kurang = [c for c in (kol, KOL_STATUS) if c not in k.columns]
        if kurang:
            raise SystemExit(
                f"Kolom {kurang} tidak ada di {path.name}: {list(k.columns)}"
            )
        k = k.dropna(subset=[kol])
        bag.append(
            pd.DataFrame(
                {
                    "entri": rapikan(k[kol]),
                    "label": rapikan(k[KOL_STATUS]).map(PETA),
                    "asal": asal,
                }
            )
        )
    k = pd.concat(bag, ignore_index=True)
    k = k[k["entri"].ne("")]
    if k["label"].isna().any():
        raise SystemExit(f"Label kamus tidak dikenal:\n{k[k['label'].isna()]}")
    k = k.groupby(["entri", "label"], as_index=False)["asal"].agg(" + ".join)
    ganda = k[k["entri"].duplicated(keep=False)]
    if len(ganda):
        raise SystemExit(f"Entri dengan label berbeda:\n{ganda}")
    k["n_kata"] = k["entri"].str.count(" ") + 1
    k["n_huruf"] = k["entri"].str.len()
    return k.sort_values(
        ["n_kata", "n_huruf", "entri"],
        ascending=[False, False, True],
        ignore_index=True,
    )


def cocokkan(teks, kamus):
    T = rapikan(teks.fillna("")).tolist()
    n = {c: np.zeros(len(T), dtype=int) for c in URUT}
    pemicu = [[] for _ in T]
    frek = []
    for e, lab in zip(kamus["entri"], kamus["label"]):
        rx = re.compile(rf"(?<!\w){re.escape(e)}(?!\w)")
        nd = nc = 0
        for i in [i for i, t in enumerate(T) if e in t]:
            T[i], k = rx.subn(" | ", T[i])
            if k:
                n[lab][i] += k
                pemicu[i].append(f"{e}[{lab}]")
                nd, nc = nd + 1, nc + k
        frek.append((nd, nc))
    return pd.DataFrame(n, index=teks.index), pemicu, frek


def nst(s):
    return round(s.map(SKOR).mean(), 4) if len(s) else np.nan


cek_awal()
kamus = muat_kamus()

df = pd.read_csv(
    SRC,
    encoding="utf-8-sig",
    usecols=[KEY, TGL, TEKS, "label_final", "sumber"],
    dtype={KEY: str},
).rename(columns={"label_final": "label_lama", "sumber": "sumber_lama"})
if df[KEY].duplicated().any():
    raise SystemExit(f"{KEY} tidak unik di {SRC.name}")
df["label_lama"] = rapikan(df["label_lama"]).where(lambda s: s.isin(URUT))

H, pemicu, frek = cocokkan(df[TEKS], kamus)
kamus["n_dokumen"] = [a for a, _ in frek]
kamus["n_cocok"] = [b for _, b in frek]

status = pd.Series(
    np.select(
        [H["pos"] > H["neg"], H["neg"] > H["pos"], H["pos"] > 0, H["neu"] > 0],
        ["dominan_pos", "dominan_neg", "imbang", "hanya_netral"],
        "tanpa_pemicu",
    ),
    index=df.index,
)
aturan = {"dominan_pos": "pos", "dominan_neg": "neg", "hanya_netral": "neu"}
if IMBANG != "lama":
    aturan["imbang"] = IMBANG
baru = status.map(aturan)
pakai = baru.notna()

df = df.assign(
    n_pos=H["pos"],
    n_neg=H["neg"],
    n_neu=H["neu"],
    status=status,
    label_final=baru.where(pakai, df["label_lama"]),
    sumber=np.where(pakai, "leksikon", df["sumber_lama"]),
    pemicu=[" ; ".join(p) for p in pemicu],
)
df["berubah_dari_final2"] = df["label_final"].ne(df["label_lama"])

keluar = df[df["label_final"].isin(URUT)].copy()
keluar[
    [
        KEY,
        TGL,
        TEKS,
        "label_final",
        "sumber",
        "status",
        "n_pos",
        "n_neg",
        "n_neu",
        "pemicu",
        "label_lama",
        "sumber_lama",
        "berubah_dari_final2",
    ]
].to_csv(OUT, index=False, encoding="utf-8-sig")

kamus.drop(columns="n_huruf").sort_values("n_dokumen", ascending=False).to_csv(
    OUT_FREK, index=False, encoding="utf-8-sig"
)

j2 = keluar["label_lama"].value_counts().reindex(URUT, fill_value=0)
j3 = keluar["label_final"].value_counts().reindex(URUT, fill_value=0)
n = len(keluar)
jumlah = pd.DataFrame(
    {
        "sentimen": [NAMA[k] for k in URUT],
        "jumlah_FINAL2": j2.to_numpy(),
        "persen_FINAL2": (j2.to_numpy() / n * 100).round(2),
        "jumlah_FINAL3": j3.to_numpy(),
        "persen_FINAL3": (j3.to_numpy() / n * 100).round(2),
    }
)
jumlah.loc[len(jumlah)] = [
    "Total",
    int(j2.sum()),
    round(j2.sum() / n * 100, 2),
    n,
    100.0,
]
jumlah.to_csv(OUT_JUMLAH, index=False, encoding="utf-8-sig")

tgl = pd.to_datetime(keluar[TGL], errors="coerce").dt.strftime("%Y-%m-%d")
keluar["periode"] = None
for nama, awal, akhir in PERIODE:
    keluar.loc[tgl.between(awal, akhir), "periode"] = nama
baris = []
for nama, g in [("Total", keluar)] + [
    (p, keluar[keluar["periode"].eq(p)]) for p, _, _ in PERIODE
]:
    baris.append(
        {
            "periode": nama,
            "n": len(g),
            "berubah": int(g["berubah_dari_final2"].sum()),
            "NSt_FINAL2": nst(g["label_lama"].dropna()),
            "NSt_FINAL3": nst(g["label_final"]),
        }
    )
periode = pd.DataFrame(baris)
periode.to_csv(OUT_PERIODE, index=False, encoding="utf-8-sig")

keluar.loc[
    keluar["berubah_dari_final2"],
    [KEY, TGL, TEKS, "label_lama", "label_final", "status", "pemicu"],
].to_csv(OUT_BERUBAH, index=False, encoding="utf-8-sig")

tanda = keluar["berubah_dari_final2"].map({True: "berubah", False: "tetap"})
print(f"Input  : {SRC}")
print(
    f"Kamus  : {len(kamus)} entri | tidak pernah cocok: {int(kamus['n_dokumen'].eq(0).sum())}"
)
print(f"IMBANG : {IMBANG}")
print(f"\nTotal dokumen          : {n}")
print(f"  Memuat pemicu kamus  : {int(keluar['status'].ne('tanpa_pemicu').sum())}")
print(f"  Label ditimpa kamus  : {int(keluar['sumber'].eq('leksikon').sum())}")
print(f"  Label berubah        : {int(keluar['berubah_dari_final2'].sum())}")
print("\nStatus pemicu dan perubahan label:")
print(pd.crosstab(keluar["status"], tanda, margins=True).to_string())
print("\nMatriks label (baris FINAL 2, kolom FINAL 3):")
print(
    pd.crosstab(keluar["label_lama"], keluar["label_final"], margins=True).to_string()
)
print("\nDistribusi sentimen:")
print(jumlah.to_string(index=False))
print("\nPerubahan per periode:")
print(periode.to_string(index=False))
if keluar["periode"].isna().any():
    print(
        f"\nPERINGATAN: {int(keluar['periode'].isna().sum())} dokumen di luar tiga periode."
    )
if n < len(df):
    print(f"\nPERINGATAN: {len(df) - n} dokumen tanpa label sah tidak disertakan.")
print(f"\nTersimpan: {OUT}")
for p in (OUT_FREK, OUT_JUMLAH, OUT_PERIODE, OUT_BERUBAH):
    print(f"Tersimpan: {p}")

PERINGATAN: FINAL 2.csv di C:\Users\EZRA\Downloads\tempat dan G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi berbeda.
            Kode ini memakai C:\Users\EZRA\Downloads\tempat\FINAL 2.csv

Input  : C:\Users\EZRA\Downloads\tempat\FINAL 2.csv
Kamus  : 962 entri | tidak pernah cocok: 65
IMBANG : neg

Total dokumen          : 14050
  Memuat pemicu kamus  : 9200
  Label ditimpa kamus  : 9200
  Label berubah        : 1608

Status pemicu dan perubahan label:
berubah_dari_final2  berubah  tetap    All
status                                    
dominan_neg              299   3261   3560
dominan_pos              675   3826   4501
hanya_netral              25    207    232
imbang                   609    298    907
tanpa_pemicu               0   4850   4850
All                     1608  12442  14050

Matriks label (baris FINAL 2, kolom FINAL 3):
label_final   neg   neu   pos    All
label_lama                          
neg          5022    14   671   5707
neu           464  2981   